# Project FORESIGHT

# Notebook 2: Data Cleaning & Preparation

## Business Context

High-quality data is essential for accurate forecasting, inventory optimization, and business reporting.

This notebook focuses on preparing clean, consistent, and analysis-ready datasets by validating data types, investigating missing values, identifying duplicate records, and performing data quality checks.

## Objectives

This notebook will:

- Load all datasets
- Convert columns to appropriate data types
- Investigate missing values
- Analyze duplicate records
- Validate business rules
- Prepare clean datasets for EDA and forecasting
- Export cleaned datasets

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
base_path = "/kaggle/input/datasets/mrayyanshehzad/synthetic-retail-dataset-10-million-transactions/retail_clean_dataset/"

customers = pd.read_csv(base_path + "customer_master.csv")
inventory = pd.read_csv(base_path + "inventory_snapshot.csv")
promotions = pd.read_csv(base_path + "promotions.csv")
sales = pd.read_csv(base_path + "sales_transactions.csv")
sku = pd.read_csv(base_path + "sku_master.csv")
stores = pd.read_csv(base_path + "store_master.csv")
flags = pd.read_csv(base_path + "sku_inventory_flags.csv")

In [3]:
# ============================================
# Convert Date Columns
# ============================================

sales["date"] = pd.to_datetime(sales["date"])

customers["registration_date"] = pd.to_datetime(
    customers["registration_date"]
)

inventory["last_restock_date"] = pd.to_datetime(
    inventory["last_restock_date"]
)

promotions["start_date"] = pd.to_datetime(
    promotions["start_date"]
)

promotions["end_date"] = pd.to_datetime(
    promotions["end_date"]
)

stores["opening_date"] = pd.to_datetime(
    stores["opening_date"]
)

flags["window_start"] = pd.to_datetime(
    flags["window_start"]
)

flags["window_end"] = pd.to_datetime(
    flags["window_end"])

In [4]:
datasets = {
    "Customers": customers,
    "Inventory": inventory,
    "Promotions": promotions,
    "Sales": sales,
    "SKU": sku,
    "Stores": stores,
    "Flags": flags
}

for name, df in datasets.items():
    print("=" * 80)
    print(name)
    print("=" * 80)
    print(df.dtypes)
    print("\n")

Customers
cust_id                      object
age                           int64
gender                       object
city                         object
loyalty_segment              object
preferred_channel            object
registration_date    datetime64[ns]
dtype: object


Inventory
store_id                     object
sku_id                       object
stock_on_hand                 int64
reorder_point                 int64
safety_stock                  int64
last_restock_date    datetime64[ns]
dtype: object


Promotions
promo_id                object
promo_name              object
start_date      datetime64[ns]
end_date        datetime64[ns]
discount_pct           float64
promo_type              object
target_type             object
target_value            object
dtype: object


Sales
date            datetime64[ns]
receipt_id              object
store_id                object
sku_id                  object
customer_id             object
quantity                 int64
unit_price   

# Missing Value Analysis

The objective of this section is to identify missing values across all datasets and determine whether they represent genuine data quality issues or valid business scenarios.

Each missing value will be evaluated before deciding on an appropriate treatment strategy.

In [5]:
def missing_value_summary(df, dataset_name):

    missing = pd.DataFrame({
        "Missing Values": df.isnull().sum(),
        "Missing Percentage": (
            df.isnull().sum() / len(df) * 100
        ).round(2)
    })

    missing = missing[missing["Missing Values"] > 0]

    print("=" * 80)
    print(dataset_name.upper())
    print("=" * 80)

    if missing.empty:
        print("✅ No Missing Values Found")
    else:
        display(missing.sort_values(
            by="Missing Percentage",
            ascending=False
        ))

In [6]:
for name, df in datasets.items():
    missing_value_summary(df, name)

CUSTOMERS
✅ No Missing Values Found
INVENTORY
✅ No Missing Values Found
PROMOTIONS
✅ No Missing Values Found
SALES


,Missing Values,Missing Percentage
promo_id,7881687,79.04


SKU
✅ No Missing Values Found
STORES
✅ No Missing Values Found
FLAGS


,Missing Values,Missing Percentage
window_start,400,66.67
window_end,400,66.67


## Missing Value Assessment

### Sales Dataset

The `promo_id` column contains approximately **79% missing values**.

After investigation, these missing values were determined to represent transactions completed without an active promotion rather than data quality issues.

Therefore:

- No imputation will be performed.
- Missing values will be retained.
- During analysis, these records will be treated as "No Promotion."

In [7]:
flags[flags["window_start"].isnull()].head(10)

,sku_id,flag,affected_stores,window_start,window_end,notes
200,SKU00721,SLOW_MOVER,ST11;ST10;ST06;ST29;ST05;ST24;ST14;ST30;ST08;S...,NaT,NaT,Chronically low-demand SKU; overstocked with a...
201,SKU03220,SLOW_MOVER,ST10;ST15;ST14;ST24;ST25;ST01;ST21;ST13;ST19;S...,NaT,NaT,Chronically low-demand SKU; overstocked with a...
202,SKU03749,SLOW_MOVER,ST26;ST29;ST23;ST12;ST14;ST21;ST03;ST08;ST11;S...,NaT,NaT,Chronically low-demand SKU; overstocked with a...
203,SKU02209,SLOW_MOVER,ST01;ST30;ST18;ST11;ST24;ST03;ST19;ST29;ST14;S...,NaT,NaT,Chronically low-demand SKU; overstocked with a...
204,SKU00170,SLOW_MOVER,ST15;ST30;ST03;ST25;ST28;ST07;ST23;ST01;ST14;S...,NaT,NaT,Chronically low-demand SKU; overstocked with a...
205,SKU00052,SLOW_MOVER,ST13;ST26;ST06;ST20;ST25;ST10;ST07;ST02;ST28;S...,NaT,NaT,Chronically low-demand SKU; overstocked with a...
206,SKU04791,SLOW_MOVER,ST08;ST05;ST16;ST29;ST30;ST09;ST19;ST13;ST24;S...,NaT,NaT,Chronically low-demand SKU; overstocked with a...
207,SKU00631,SLOW_MOVER,ST16;ST26;ST14;ST15;ST08;ST07;ST29;ST13;ST19;S...,NaT,NaT,Chronically low-demand SKU; overstocked with a...
208,SKU01428,SLOW_MOVER,ST03;ST21;ST29;ST23;ST30;ST06;ST17;ST10;ST01;S...,NaT,NaT,Chronically low-demand SKU; overstocked with a...
209,SKU02728,SLOW_MOVER,ST04;ST21;ST10;ST20;ST16;ST14;ST03;ST17;ST28;S...,NaT,NaT,Chronically low-demand SKU; overstocked with a...


In [8]:
flags.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   sku_id           600 non-null    object        
 1   flag             600 non-null    object        
 2   affected_stores  600 non-null    object        
 3   window_start     200 non-null    datetime64[ns]
 4   window_end       200 non-null    datetime64[ns]
 5   notes            600 non-null    object        
dtypes: datetime64[ns](2), object(4)
memory usage: 28.3+ KB


## SKU Inventory Flags Dataset

The `window_start` and `window_end` columns contain missing values for approximately **66.67%** of the records.

### Business Interpretation

Further investigation showed that these missing values occur exclusively for products classified as **SLOW_MOVER**.

Unlike **STOCKOUT_RISK**, slow-moving products do not have a defined monitoring window.

Therefore, the missing dates represent valid business logic rather than incomplete data.

### Cleaning Decision

- No rows will be removed.
- Missing values will not be imputed.
- The missing values will be retained because they accurately represent the business process.

# 6. Duplicate Analysis

## Objective

The purpose of this section is to identify duplicate records across the datasets and determine whether they represent genuine data quality issues or valid business transactions.

Duplicate records will be investigated before any removal is performed.

In [9]:
def duplicate_summary(df, dataset_name):

    duplicate_count = df.duplicated().sum()
    duplicate_percentage = round(
        duplicate_count / len(df) * 100, 4
    )

    print("=" * 80)
    print(dataset_name.upper())
    print("=" * 80)

    print(f"Total Rows: {len(df):,}")
    print(f"Duplicate Rows: {duplicate_count:,}")
    print(f"Duplicate Percentage: {duplicate_percentage}%")
    print()

In [10]:
for name, df in datasets.items():
    duplicate_summary(df, name)

CUSTOMERS
Total Rows: 10,000
Duplicate Rows: 0
Duplicate Percentage: 0.0%

INVENTORY
Total Rows: 21,228
Duplicate Rows: 0
Duplicate Percentage: 0.0%

PROMOTIONS
Total Rows: 100
Duplicate Rows: 0
Duplicate Percentage: 0.0%

SALES
Total Rows: 9,972,038
Duplicate Rows: 13,019
Duplicate Percentage: 0.1306%

SKU
Total Rows: 5,000
Duplicate Rows: 0
Duplicate Percentage: 0.0%

STORES
Total Rows: 30
Duplicate Rows: 0
Duplicate Percentage: 0.0%

FLAGS
Total Rows: 600
Duplicate Rows: 0
Duplicate Percentage: 0.0%



# Duplicate Investigation

The duplicate analysis identified duplicate records in the Sales dataset.

Before removing these records, the duplicate transactions will be investigated to determine whether they represent genuine duplicate records or valid business transactions.

In [11]:
sales_duplicates = sales[
    sales.duplicated(keep=False)
].sort_values(
    by=["receipt_id", "sku_id"]
)

print(f"Rows belonging to duplicate groups: {len(sales_duplicates):,}")

display(sales_duplicates.head(20))

Rows belonging to duplicate groups: 25,884


,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id
270,2025-05-11,RCPT00000149,ST10,SKU02141,CUST09365,2,471.88,943.76,Online,0.0,NaN
272,2025-05-11,RCPT00000149,ST10,SKU02141,CUST09365,2,471.88,943.76,Online,0.0,NaN
314,2022-08-03,RCPT00000169,ST29,SKU04321,CUST07066,1,961.30,961.30,In-Store,0.0,NaN
315,2022-08-03,RCPT00000169,ST29,SKU04321,CUST07066,1,961.30,961.30,In-Store,0.0,NaN
544,2025-11-14,RCPT00000289,ST06,SKU04321,CUST02146,1,961.30,961.30,In-Store,0.0,NaN
545,2025-11-14,RCPT00000289,ST06,SKU04321,CUST02146,1,961.30,961.30,In-Store,0.0,NaN
2945,2024-06-22,RCPT00001531,ST25,SKU04321,CUST01969,1,961.30,744.05,Online,22.6,PROMO045
2946,2024-06-22,RCPT00001531,ST25,SKU04321,CUST01969,1,961.30,744.05,Online,22.6,PROMO045
5864,2025-10-09,RCPT00003040,ST13,SKU04321,CUST08409,1,961.30,961.30,Mobile App,0.0,NaN
5865,2025-10-09,RCPT00003040,ST13,SKU04321,CUST08409,1,961.30,961.30,Mobile App,0.0,NaN


In [12]:
sales_duplicates.nunique()

date             1460
receipt_id      12857
store_id           30
sku_id            439
customer_id      6879
quantity            5
unit_price        438
total_value       971
channel             3
discount_pct       52
promo_id           55
dtype: int64

In [13]:
sales_duplicates.describe(include="all")

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id
count,25884,25884,25884,25884,25884,25884.000000,25884.000000,25884.000000,25884,25884.000000,5402
unique,NaN,12857,30,439,6879,NaN,NaN,NaN,3,NaN,55
top,NaN,RCPT00456514,ST21,SKU04321,CUST01706,NaN,NaN,NaN,In-Store,NaN,PROMO075
freq,NaN,4,1274,20872,17,NaN,NaN,NaN,14276,NaN,484
mean,2024-02-03 22:25:48.817802496,NaN,NaN,NaN,NaN,1.370499,885.349402,1139.750939,NaN,6.245588,NaN
min,2022-01-01 00:00:00,NaN,NaN,NaN,NaN,1.000000,34.080000,34.080000,NaN,0.000000,NaN
25%,2023-02-03 00:00:00,NaN,NaN,NaN,NaN,1.000000,961.300000,825.760000,NaN,0.000000,NaN
50%,2024-02-12 00:00:00,NaN,NaN,NaN,NaN,1.000000,961.300000,961.300000,NaN,0.000000,NaN
75%,2025-01-20 00:00:00,NaN,NaN,NaN,NaN,2.000000,961.300000,1037.900000,NaN,0.000000,NaN
max,2025-12-31 00:00:00,NaN,NaN,NaN,NaN,5.000000,4181.580000,9808.280000,NaN,49.800000,NaN


# Duplicate Assessment

## Observation

The Sales dataset contains **13,019 duplicate records**.

Further investigation showed that the duplicate groups contain identical transaction information, including:

- Receipt ID
- SKU ID
- Customer ID
- Quantity
- Unit Price
- Total Value
- Promotion ID

These records appear to be exact duplicate transaction rows rather than legitimate repeated purchases.

## Cleaning Decision

- Duplicate rows will be removed from the Sales dataset.
- Other datasets do not contain duplicate records.
- Removing these duplicate rows will improve data quality without affecting business logic.

In [14]:
# Remove duplicate records

sales = sales.drop_duplicates()

print("Duplicate rows removed successfully.")

print(f"Remaining rows: {len(sales):,}")

Duplicate rows removed successfully.
Remaining rows: 9,959,019


In [15]:
sales.duplicated().sum()

np.int64(0)

# 7. Data Validation

## Objective

After cleaning the datasets, a series of business validation checks are performed to ensure the data is logically consistent and suitable for analysis and forecasting.

The validation focuses on sales quantities, pricing, discount values, transaction amounts, and key identifiers.

In [16]:
def validate_sales_data(df):

    print("=" * 80)
    print("SALES DATA VALIDATION")
    print("=" * 80)

    print("\n1. Quantity <= 0")
    print((df["quantity"] <= 0).sum())

    print("\n2. Unit Price <= 0")
    print((df["unit_price"] <= 0).sum())

    print("\n3. Total Value <= 0")
    print((df["total_value"] <= 0).sum())

    print("\n4. Discount Outside 0-100")
    print(((df["discount_pct"] < 0) | (df["discount_pct"] > 100)).sum())

    print("\n5. Missing Customer IDs")
    print(df["customer_id"].isnull().sum())

    print("\n6. Missing SKU IDs")
    print(df["sku_id"].isnull().sum())

    print("\n7. Missing Store IDs")
    print(df["store_id"].isnull().sum())

In [17]:
validate_sales_data(sales)

SALES DATA VALIDATION

1. Quantity <= 0
0

2. Unit Price <= 0
0

3. Total Value <= 0
0

4. Discount Outside 0-100
0

5. Missing Customer IDs
0

6. Missing SKU IDs
0

7. Missing Store IDs
0


# Validation Assessment

## Findings

The Sales dataset successfully passed all validation checks.

### Validation Results

- No invalid sales quantities were detected.
- No invalid unit prices were identified.
- No negative transaction values were found.
- All discount percentages fall within the expected business range.
- No missing customer, SKU, or store identifiers were identified.

### Conclusion

The Sales dataset is considered clean, internally consistent, and ready for exploratory data analysis and predictive modeling.

In [18]:
import os

output_path = "/kaggle/working/clean_data"

os.makedirs(output_path, exist_ok=True)

customers.to_csv(f"{output_path}/customers_clean.csv", index=False)
inventory.to_csv(f"{output_path}/inventory_clean.csv", index=False)
promotions.to_csv(f"{output_path}/promotions_clean.csv", index=False)
sales.to_csv(f"{output_path}/sales_clean.csv", index=False)
sku.to_csv(f"{output_path}/sku_clean.csv", index=False)
stores.to_csv(f"{output_path}/stores_clean.csv", index=False)
flags.to_csv(f"{output_path}/flags_clean.csv", index=False)

print("Clean datasets exported successfully.")

Clean datasets exported successfully.


# Notebook Summary

## Completed Tasks

- Loaded all datasets.
- Converted date columns to datetime format.
- Performed missing value analysis.
- Investigated missing values using business context.
- Investigated duplicate records.
- Removed duplicate sales transactions.
- Validated the cleaned Sales dataset.
- Exported cleaned datasets for downstream analysis.

## Next Steps

The cleaned datasets generated in this notebook will be used in **Notebook 3: Exploratory Data Analysis (EDA)** to identify sales trends, customer behavior, product performance, promotion effectiveness, and inventory insights.